# Generate Person And Condition

This notebook demonstrates the updated schema-building functions.

It shows how to:
- sample a routed graph state from a profile, explicitly selecting a case with a non-`none` biomarker when available
- build a `Person` and `Condition_Occurrence` without forcing treatment assignment
- build a full case bundle including `Treatment_Episode`
- inspect the rich domain `model_dump()` output for the person and condition objects
- inspect the final OMOP-facing rows produced by `to_omop_row()` and `to_omop_bundle()`


In [1]:
from pathlib import Path
import sys
import random

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

from scoop.configuration.config_loader import load_routed_profile
from scoop.schema.object_factories import (
    build_case_bundle,
    build_condition_occurrence,
    build_person,
)


def omop_row_frame(row):
    return pd.DataFrame([
        {"field": column.name, "value": getattr(row, column.name)}
        for column in row.__table__.columns
    ])


def omop_rows_frame(rows):
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame([
        {column.name: getattr(row, column.name) for column in row.__table__.columns}
        for row in rows
    ])


2026-04-06 21:58:10,604 | DEBUG   | scoop | SCOOP logging initialized


In [2]:
profile_id = "nsclc"
runtime = load_routed_profile(profile_id, base_dir=CONFIG_ROOT)

random.seed(17)
for _ in range(5000):
    routed = runtime.sample_routed_patient()
    biomarker = routed.final_sample.get("condition_occurrence.biomarker")
    if routed.graph_outcome is not None and biomarker not in {None, "none"}:
        break
else:
    raise RuntimeError("Did not sample a routed patient with both a treatment outcome and a non-`none` biomarker within 5000 draws.")

display(Markdown(f"## Routed Sample From `{profile_id}` With A Valid Biomarker"))
display(pd.DataFrame([{
    "pathway": routed.pathway,
    "tail_graph_name": routed.tail_graph_name,
    "primary_category_id": routed.graph_outcome.primary_category_id,
    "active_endpoints": routed.graph_outcome.active_endpoints,
}]))

preview_keys = [
    key for key in routed.final_sample
    if key.startswith("person.") or key.startswith("condition_occurrence.") or key.startswith("condition_episode.") or key.startswith("treatment_episode.")
]
display(pd.DataFrame([
    {"field": key, "value": routed.final_sample[key]}
    for key in preview_keys
    if routed.final_sample[key] is not None
]))


## Routed Sample From `nsclc` With A Valid Biomarker

,pathway,tail_graph_name,primary_category_id,active_endpoints
0,nsclc_stage4_nodrivers,nsclc_stg4_nodrivers,ici_monotherapy_or_ici_with_chemotherapy_nonsq...,(ici_montherapy_or_ici_with_chemotherapy_nonsq...


,field,value
0,person.age,55
1,person.year_of_diagnosis,2018
2,person.years_since_diagnosis,2
3,person.sex,male
4,person.smoking_history,current
5,person.weight,85.768256
6,condition_occurrence.laterality,right
7,condition_occurrence.n_stage,N0
8,condition_occurrence.t_8th,T2a
9,condition_occurrence.m_stage,M1


In [3]:
person = build_person(routed.final_sample, attach_condition=False)
condition = build_condition_occurrence(
    person,
    routed.final_sample,
    assign_treatment=False,
)
person.condition = condition

display(Markdown("## Explicit Person + Condition Without Treatment"))
display(pd.DataFrame([
    {
        "person_id": person.person_id,
        "sex": person.sex,
        "age": person.age,
        "condition_occurrence_id": condition.condition_occurrence_id,
        "condition_start_date": condition.condition_start_date,
        "cancer_stage": condition.cancer_stage,
        "biomarker": condition.biomarker,
        "overarching_episode_generated": condition.overarching_episode is not None,
        "regimen_attached": condition.regimen is not None,
    }
]))

display(Markdown("## Selected Biomarker Review"))
display(pd.DataFrame([{
    "biomarker": condition.biomarker,
    "condition_concept_id": condition.condition_concept_id,
    "condition_source_value": condition.condition_source_value,
}]))


2026-04-06 21:58:11,622 | DEBUG   | scoop.schema.person | Initialised Person[1] age=55 sex=male


## Explicit Person + Condition Without Treatment

,person_id,sex,age,condition_occurrence_id,condition_start_date,cancer_stage,biomarker,overarching_episode_generated,regimen_attached
0,1,male,55,1,2018-06-23,stage_iv,kras,True,False


## Selected Biomarker Review

,biomarker,condition_concept_id,condition_source_value
0,kras,44501404,8140/3-C34.3


In [4]:
condition_episode = condition.overarching_episode
episode_dump = condition_episode.model_dump(exclude={"condition_occurrence"})
episode_row = condition_episode.to_omop_row()
episode_bundle = condition_episode.to_omop_bundle()

display(Markdown("## Condition Episode Summary"))
display(pd.DataFrame([{
    "episode_id": condition_episode.episode_id,
    "episode_concept_id": condition_episode.episode_concept_id,
    "episode_type_concept_id": condition_episode.episode_type_concept_id,
    "episode_object_concept_id": condition_episode.episode_object_concept_id,
    "episode_source_value": condition_episode.episode_source_value,
    "episode_start_date": condition_episode.episode_start_date,
    "linked_condition_occurrence_id": condition_episode.condition_occurrence_id,
    "matches_condition_start_date": condition_episode.episode_start_date == condition.condition_start_date,
    "event_links": len(condition_episode.episode_events),
}]))

display(Markdown("## Condition Episode Domain Object"))
display(pd.DataFrame([
    {"field": key, "value": value}
    for key, value in episode_dump.items()
    if key != "episode_events"
]))

display(Markdown("## Condition Episode OMOP Row"))
display(omop_row_frame(episode_row))

display(Markdown("## Episode Event Links"))
display(pd.DataFrame(episode_dump.get("episode_events", [])))

display(Markdown("## Episode Event OMOP Rows"))
display(omop_rows_frame(episode_bundle.episode_events))


## Condition Episode Summary

,episode_id,episode_concept_id,episode_type_concept_id,episode_object_concept_id,episode_source_value,episode_start_date,linked_condition_occurrence_id,matches_condition_start_date,event_links
0,1,32528,32533,44501404,eposode_of_care,2018-06-23,1,True,1


## Condition Episode Domain Object

,field,value
0,person_id,1
1,episode_id,1
2,episode_parent_id,None
3,episode_start_date,2018-06-23
4,episode_start_datetime,2018-06-23 00:00:00
5,episode_end_date,None
6,episode_end_datetime,None
7,episode_number,None
8,episode_source_value,eposode_of_care
9,episode_concept_id,32528


## Condition Episode OMOP Row

,field,value
0,episode_id,1
1,episode_parent_id,None
2,episode_start_date,2018-06-23
3,episode_start_datetime,2018-06-23 00:00:00
4,episode_end_date,None
5,episode_end_datetime,None
6,episode_number,None
7,episode_source_value,eposode_of_care
8,episode_concept_id,32528
9,episode_object_concept_id,44501404


## Episode Event Links

,episode_id,event_id,episode_event_field_concept_id
0,1,1,1147127


## Episode Event OMOP Rows

,episode_id,event_id,episode_event_field_concept_id
0,1,1,1147127


In [5]:
bundle_without_treatment = build_case_bundle(
    graph_sample=routed.final_sample,
    include_treatment_episode=False,
)

bundle_with_treatment = build_case_bundle(
    graph_sample=routed.final_sample,
    include_treatment_episode=True,
)

display(Markdown("## Case Bundle Comparison"))
display(pd.DataFrame([
    {
        "bundle": "without_treatment",
        "person_id": bundle_without_treatment.person.person_id,
        "condition_occurrence_id": bundle_without_treatment.condition_occurrence.condition_occurrence_id,
        "treatment_episode": bundle_without_treatment.treatment_episode is not None,
    },
    {
        "bundle": "with_treatment",
        "person_id": bundle_with_treatment.person.person_id,
        "condition_occurrence_id": bundle_with_treatment.condition_occurrence.condition_occurrence_id,
        "treatment_episode": bundle_with_treatment.treatment_episode is not None,
    },
]))


2026-04-06 21:58:13,439 | DEBUG   | scoop.schema.person | Initialised Person[2] age=55 sex=male
2026-04-06 21:58:13,440 | DEBUG   | scoop.schema.person | Initialised Person[3] age=55 sex=male
2026-04-06 21:58:13,441 | DEBUG   | scoop.schema.object_factories | Assigning regimen 'ici_monotherapy_or_ici_with_chemotherapy_nonsquamous' for Condition[3] from graph endpoint 'ici_montherapy_or_ici_with_chemotherapy_nonsquamous'


## Case Bundle Comparison

,bundle,person_id,condition_occurrence_id,treatment_episode
0,without_treatment,2,2,False
1,with_treatment,3,3,True


In [6]:
treatment = bundle_with_treatment.treatment_episode

display(Markdown("## Treatment Episode Summary"))
display(pd.DataFrame([{
    "regimen_type": treatment.regimen_type,
    "regimen_category_id": treatment.regimen_category_id,
    "graph_endpoint": treatment.graph_endpoint,
    "graph_alias_used": treatment.graph_alias_used,
    "registry_resolver_type": treatment.registry_resolver_type,
    "source_profile": treatment.source_profile,
}]))

display(pd.DataFrame([
    {"modifier": key, "value": value}
    for key, value in sorted(treatment.decision_context.items())
]))


## Treatment Episode Summary

,regimen_type,regimen_category_id,graph_endpoint,graph_alias_used,registry_resolver_type,source_profile
0,ici_monotherapy_or_ici_with_chemotherapy_nonsq...,ici_monotherapy_or_ici_with_chemotherapy_nonsq...,ici_montherapy_or_ici_with_chemotherapy_nonsqu...,ici_montherapy_or_ici_with_chemotherapy_nonsqu...,composite,nsclc_stg4_nodrivers


,modifier,value
0,condition_episode.chemotherapy_eligibility,chemo_yes
1,condition_episode.group_stage,metastatic
2,condition_episode.key_mutation,km_no
3,condition_episode.pathway,nsclc_stage4_nodrivers
4,condition_episode.radiotherapy_eligibility,rt_yes
5,condition_episode.resectability,inoperable
6,condition_episode.targetable_mutation,tm_no
7,condition_occurrence.biomarker,kras
8,condition_occurrence.cancer_stage,stage_iv
9,condition_occurrence.laterality,right


In [7]:
person_dump = bundle_with_treatment.person.model_dump(exclude={"condition"})
condition_dump = bundle_with_treatment.condition_occurrence.model_dump(
    exclude={"person", "regimen"}
)
omop_bundle = bundle_with_treatment.person.to_omop_bundle(include_related=True)

display(Markdown("## Person `model_dump()`"))
display(pd.DataFrame([
    {"field": key, "value": value}
    for key, value in person_dump.items()
    if key not in {"measurements", "observations"}
]))

display(Markdown("## Person Measurements"))
display(pd.DataFrame(person_dump.get("measurements", [])))

display(Markdown("## Person Observations"))
display(pd.DataFrame(person_dump.get("observations", [])))

display(Markdown("## Condition `model_dump()`"))
display(pd.DataFrame([
    {"field": key, "value": value}
    for key, value in condition_dump.items()
    if key not in {"measurements", "observations"}
]))

display(Markdown("## Condition Measurements"))
display(pd.DataFrame(condition_dump.get("measurements", [])))

display(Markdown("## Condition Observations"))
display(pd.DataFrame(condition_dump.get("observations", [])))

display(Markdown("## OMOP Bundle Counts"))
display(pd.DataFrame([{
    "condition_occurrences": len(omop_bundle.condition_occurrences),
    "episodes": len(omop_bundle.episodes),
    "episode_events": len(omop_bundle.episode_events),
    "measurements": len(omop_bundle.measurements),
    "observations": len(omop_bundle.observations),
}]))

display(Markdown("## OMOP Person Row"))
display(omop_row_frame(omop_bundle.person))

display(Markdown("## OMOP Condition Rows"))
display(omop_rows_frame(omop_bundle.condition_occurrences))

display(Markdown("## OMOP Episode Rows"))
display(omop_rows_frame(omop_bundle.episodes))

display(Markdown("## OMOP Episode Event Rows"))
display(omop_rows_frame(omop_bundle.episode_events))

display(Markdown("## OMOP Measurement Rows"))
display(omop_rows_frame(omop_bundle.measurements))

display(Markdown("## OMOP Observation Rows"))
display(omop_rows_frame(omop_bundle.observations))


## Person `model_dump()`

,field,value
0,person_id,3
1,race_concept_id,0
2,ethnicity_concept_id,0
3,year_of_birth,1971
4,gender_concept_id,8507


## Person Measurements

,person_id,measurement_id,measurement_concept_id,measurement_date,measurement_event_id,meas_event_field_concept_id,unit_concept_id,value_as_concept_id,value_as_number,value_source_value
0,3,1,4099154,2026-04-06,None,None,9529.0,None,85.768256,85.76825588289049
1,3,2,36305384,2026-04-06,None,None,NaN,None,2.000000,2


## Person Observations

...

,episode_id,event_id,episode_event_field_concept_id
0,3,3,1147127


## OMOP Measurement Rows

,measurement_id,person_id,measurement_concept_id,measurement_date,measurement_datetime,measurement_time,measurement_type_concept_id,operator_concept_id,unit_concept_id,range_low,...,visit_detail_id,measurement_source_value,measurement_source_concept_id,unit_source_value,unit_source_concept_id,value_source_value,measurement_event_id,meas_event_field_concept_id,value_as_number,value_as_concept_id
0,8,3,4099154,2026-04-06,2026-04-06,None,None,None,9529.0,None,...,None,None,None,None,None,85.76825588289049,NaN,NaN,85.768256,NaN
1,9,3,36305384,2026-04-06,2026-04-06,None,None,None,NaN,None,...,None,None,None,None,None,2,NaN,NaN,2.000000,NaN
2,10,3,35226255,2026-04-06,2026-04-06,None,None,None,NaN,None,...,None,None,None,None,None,OMOP5031807,3.0,1147127.0,NaN,NaN
3,11,3,4180182,2026-04-06,2026-04-06,None,None,None,NaN,None,...,None,None,None,None,None,kras,NaN,NaN,NaN,35948791.0
4,12,3,1092026,2026-04-06,2026-04-06,None,None,None,8554.0,None,...,None,None,None,None,None,high,NaN,NaN,75.000000,NaN
5,13,3,734320,2026-04-06,2026-04-06,None,None,None,NaN,None,...,None,None,None,None,None,stage_iv,3.0,1147127.0,NaN,1633577.0
6,14,3,35918306,2026-04-06,2026-04-06,None,None,None,NaN,None,...,None,None,None,None,None,right,3.0,1147127.0,NaN,45881626.0


## OMOP Observation Rows

,observation_id,person_id,observation_concept_id,observation_date,observation_datetime,observation_type_concept_id,value_as_string,qualifier_concept_id,unit_concept_id,provider_id,...,visit_detail_id,observation_source_value,observation_source_concept_id,unit_source_value,qualifier_source_value,value_source_value,observation_event_id,obs_event_field_concept_id,value_as_number,value_as_concept_id
0,2,3,43054909,2026-04-06,2026-04-06,None,None,None,None,None,...,None,current,None,None,None,None,None,None,None,45881517


In [8]:
{
    "domain_condition_dump": condition.model_dump(exclude={"person", "regimen"}),
    "omop_condition_row_type": type(condition.to_omop_row()).__name__,
    "omop_episode_row_type": type(condition.overarching_episode.to_omop_row()).__name__,
}


{'domain_condition_dump': {'person_id': 1,
  'condition_occurrence_id': 1,
  'condition_concept_id': 44501404,
  'condition_source_concept_id': 44501404,
  'condition_source_value': '8140/3-C34.3',
  'year_of_diagnosis': 2018,
  'years_since_diagnosis': 2,
  'condition_start_date': datetime.date(2018, 6, 23),
  'condition_domain': 'nsclc',
  'morphology': 'nonsquamous',
  'laterality': 'right',
  'topog': 'lower',
  'stage': 'stage_iv',
  'cancer_stage': 'stage_iv',
  'biomarker': 'kras',
  'pdl1': 'high',
  'source_profile': 'nsclc_stg4_nodrivers',
  'measurements': [{'person_id': 1,
    'measurement_id': 15,
    'measurement_concept_id': 35226268,
    'measurement_date': datetime.date(2026, 4, 6),
    'measurement_event_id': 1,
    'meas_event_field_concept_id': 1147127,
    'unit_concept_id': None,
    'value_as_concept_id': None,
    'value_as_number': None,
    'value_source_value': 'OMOP5031820'},
   {'person_id': 1,
    'measurement_id': 16,
    'measurement_concept_id': 4180182